# Week 3 Hands-On Lab: Exploratory Data Analysis on the Titanic Dataset

**Difficulty:** ⭐⭐⭐ &nbsp;&nbsp;|&nbsp;&nbsp; **Estimated Time:** 3–4 hours

---

## What This Lab Is

This is the **capstone lab for Week 3**. You will work through a complete, end-to-end Exploratory Data Analysis (EDA) pipeline on the Titanic dataset — the same kind of work a data scientist does before touching a model. Every section connects directly to the statistical and Python tools covered in notebooks 01–11.

---

## Learning Objectives

By the end of this lab you will be able to:

1. **Compute and interpret descriptive statistics** (mean, median, skewness, kurtosis) and explain why each matters for ML feature engineering.
2. **Detect and reason about outliers** using both the IQR method and z-scores, and make a principled decision about whether to keep or remove them.
3. **Run and interpret two-sample t-tests** including p-values, effect sizes (Cohen's d), and confidence intervals.
4. **Build confidence intervals** for proportions and means, and compare groups visually using forest-plot-style charts.
5. **Analyse correlations** with a heatmap and scatter plots, and distinguish correlation from causation.
6. **Apply NumPy vectorised operations and Pandas groupby/apply** workflows to prepare data for scikit-learn.

---

## Dataset

**Titanic passenger manifest** — 891 rows, 15 columns. Loaded directly from seaborn's built-in dataset cache. The target variable is `survived` (0 = did not survive, 1 = survived).

---

## Lab Structure

| Section | Topic | Notebooks Covered |
|---------|-------|-------------------|
| 0 | Setup & First Look | All |
| 1 | Descriptive Statistics | 01, 02 |
| 2 | Outlier Detection | 02 |
| 3 | Hypothesis Testing | 04, 05 |
| 4 | Confidence Intervals | 06 |
| 5 | Correlation Analysis | 07 |
| 6 | Visualisation Gallery | 10 |
| 7 | NumPy & Pandas Operations | 08, 09 |
| 8 | Summary Findings & ML Implications | 11 |
| 9 | Self-Check Questions | All |

---
# Section 0 — Setup & First Look

In [ ]:
# ── Core numerical & data libraries ──────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation libraries ───────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Statistical testing ───────────────────────────────────────────────────────
from scipy import stats

# ── Display helpers ───────────────────────────────────────────────────────────
from IPython.display import display

# ── Reproducibility seed (set once here, referenced throughout) ───────────────
np.random.seed(42)

# ── Global plot theme: whitegrid keeps grid lines, husl palette is colourblind-
#    friendlier than the default muted palette ─────────────────────────────────
sns.set_theme(style='whitegrid', palette='husl')

# Slightly larger default figure text for readability in the notebook
plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})

print('All libraries imported successfully.')

In [ ]:
# ── Load the Titanic dataset from seaborn's built-in cache ───────────────────
# seaborn downloads the CSV from GitHub on first call and caches it locally.
df = sns.load_dataset('titanic')

# ── Basic shape and column overview ──────────────────────────────────────────
print(f'Dataset shape: {df.shape}')          # (rows, columns)
print(f'\nColumn names and dtypes:')
print(df.dtypes)                              # object = string/categorical

print(f'\nFirst 10 rows:')
display(df.head(10))                          # use display() not print() for DataFrames

In [ ]:
# ── .info() shows non-null counts — the fastest way to spot missing data ─────
print('=== df.info() ===')
df.info()

print('\n=== df.describe() — numeric columns ===')
# include='all' would also summarise categoricals; here we keep it numeric
display(df.describe().round(2))

print('\n=== df.describe() — object / categorical columns ===')
display(df.describe(include='object').T)      # transpose so column names are rows

In [ ]:
# ── Missing-value audit ───────────────────────────────────────────────────────
# Count absolute missing and convert to a percentage of total rows
missing_counts = df.isnull().sum()                    # absolute counts
missing_pct    = (missing_counts / len(df)) * 100     # percentage

# Build a summary DataFrame, sorted descending by missing %
missing_df = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_pct':   missing_pct.round(1)
}).sort_values('missing_pct', ascending=False)

# Only show columns that actually have missing values
missing_df = missing_df[missing_df['missing_count'] > 0]
print('Columns with missing values:')
display(missing_df)

# ── Bar chart of missing % per column ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

bars = ax.bar(
    missing_df.index,
    missing_df['missing_pct'],
    color=sns.color_palette('husl', len(missing_df))
)

# Annotate each bar with its exact percentage
for bar, val in zip(bars, missing_df['missing_pct']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{val}%',
        ha='center', va='bottom', fontsize=10
    )

ax.set_title('Missing Data Audit — Titanic Dataset', fontsize=14, fontweight='bold')
ax.set_xlabel('Column')
ax.set_ylabel('Missing (%)')
ax.set_ylim(0, 105)   # leave room for text annotations above tallest bar
plt.tight_layout()
plt.show()

### Missing Data Commentary

- **`age` (~20% missing):** A substantial chunk. Random missingness is unlikely — lower-class passengers had less documentation. Imputing with the median (rather than mean) is safer because `age` is right-skewed. This is covered in Section 8.
- **`deck` (~77% missing):** Too many gaps to be useful without significant imputation. In an ML pipeline we would likely drop this column or use it as a binary "deck known / unknown" indicator.
- **`embark_town` / `embarked` (2 rows):** Negligible — safe to drop those two rows or fill with the mode.

Missing data is not just a cleaning problem — the *pattern* of missingness often carries information. A model that ignores why data is missing can learn spurious correlations.

---
# Section 1 — Descriptive Statistics
*Connects to: Notebooks 01 (Descriptive Statistics) & 02 (Measures of Spread)*

In [ ]:
# ── Central tendency: mean, median, mode for age, fare, pclass ───────────────
# dropna() is essential here — mean/median skip NaN by default in pandas,
# but being explicit reminds readers that missing values exist.

cols_of_interest = ['age', 'fare', 'pclass']
results = []

for col in cols_of_interest:
    series = df[col].dropna()
    results.append({
        'column':  col,
        'mean':    round(series.mean(), 2),
        'median':  round(series.median(), 2),
        'mode':    series.mode()[0],           # .mode() returns a Series; take the first
        'range':   round(series.max() - series.min(), 2),
        'IQR':     round(series.quantile(0.75) - series.quantile(0.25), 2)
    })

central_df = pd.DataFrame(results).set_index('column')
print('Central Tendency & Range Metrics')
display(central_df)

### Interpretation — Central Tendency

- **`age`:** Mean ≈ 29.7, median ≈ 28. These are close, indicating a roughly symmetric distribution with a slight right tail (older passengers pull the mean up). The IQR spans roughly 20 years, covering most adult passengers.
- **`fare`:** Mean >> median (e.g., ~32 vs ~14). This large gap is the hallmark of a **right-skewed distribution** — a small number of first-class passengers paid enormous fares, inflating the mean. For ML, the median is a more robust summary of a 'typical' ticket price.
- **`pclass`:** Mode = 3, meaning most passengers were in third class. The range is only 2 (values 1–3), so treating it as numeric makes sense, but remember: the numeric gap between classes is not necessarily equal in social/economic terms.

In [ ]:
# ── Measures of spread: variance, std, skewness, kurtosis for age & fare ─────
# Skewness > 0 → right tail; Kurtosis > 0 (excess) → heavier tails than normal

spread_results = []
for col in ['age', 'fare']:
    series = df[col].dropna()
    spread_results.append({
        'column':    col,
        'variance':  round(series.var(), 2),
        'std_dev':   round(series.std(), 2),
        'skewness':  round(series.skew(), 3),     # pandas uses Fisher's definition
        'kurtosis':  round(series.kurtosis(), 3)  # excess kurtosis (normal = 0)
    })

spread_df = pd.DataFrame(spread_results).set_index('column')
print('Measures of Spread')
display(spread_df)

# ── Quick print of IQR for completeness ──────────────────────────────────────
for col in ['age', 'fare']:
    q1, q3 = df[col].quantile([0.25, 0.75])
    print(f'{col}: Q1={q1:.2f}, Q3={q3:.2f}, IQR={q3-q1:.2f}')

### Interpretation — Spread

- **`age` skewness ≈ 0.39:** Mild positive skew — the distribution leans slightly right (a few elderly passengers). Kurtosis near 0 means tails are roughly normal.
- **`fare` skewness > 4:** Extreme right skew. A handful of fares above £200–500 massively distort the distribution. Standard deviation is misleading here — the IQR is the honest spread measure.
- **`fare` kurtosis >> 0:** Very heavy right tail (leptokurtic). This matters for ML: algorithms like linear regression assume residuals are roughly normal. Raw `fare` as a feature will push the model to fit those extreme values.

**Action:** Log-transform `fare` before using it as an ML feature. We do this next.

In [ ]:
# ── Log-transform fare and compare distributions side by side ─────────────────
# log1p(x) = log(x + 1) avoids log(0) errors for any fare = 0
df['log_fare'] = np.log1p(df['fare'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: original fare distribution ---
axes[0].hist(
    df['fare'].dropna(), bins=50,
    color='steelblue', edgecolor='white', alpha=0.85
)
axes[0].set_title('Fare — Original (Right-Skewed)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Fare (£)')
axes[0].set_ylabel('Count')
# Annotate with skewness so the viewer can put a number to the shape
axes[0].text(
    0.97, 0.95,
    f"Skewness = {df['fare'].skew():.2f}",
    transform=axes[0].transAxes,
    ha='right', va='top',
    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5)
)

# --- Right: log-transformed fare ---
axes[1].hist(
    df['log_fare'].dropna(), bins=50,
    color='darkorange', edgecolor='white', alpha=0.85
)
axes[1].set_title('log(Fare + 1) — After Log Transform', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log(Fare + 1)')
axes[1].set_ylabel('Count')
axes[1].text(
    0.97, 0.95,
    f"Skewness = {df['log_fare'].skew():.2f}",
    transform=axes[1].transAxes,
    ha='right', va='top',
    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5)
)

plt.suptitle('Effect of Log Transformation on Fare', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Fare  — skewness before: {df['fare'].skew():.3f}")
print(f"Fare  — skewness after:  {df['log_fare'].skew():.3f}")

### Why the Log Transform Matters for ML

After log-transforming, the skewness drops from ~4+ to near 0. The distribution now looks approximately normal — which is a requirement (or at least a strong preference) for:

- **Linear / logistic regression:** assumes features are roughly Gaussian.
- **Distance-based algorithms (KNN, SVM):** extreme raw values in `fare` would dominate the distance calculation.
- **Regularisation (Lasso/Ridge):** penalty terms assume features are on comparable scales.

**Rule of thumb:** if `skewness > 1`, consider a log or Box-Cox transform before modelling.

---
# Section 2 — Outlier Detection
*Connects to: Notebook 02 (Measures of Spread)*

In [ ]:
# ── Method 1: IQR Rule on `fare` ──────────────────────────────────────────────
# Any point below Q1 - 1.5*IQR or above Q3 + 1.5*IQR is flagged as an outlier.
# This is the same rule Tukey's boxplot uses.

fare_series = df['fare'].dropna()

Q1   = fare_series.quantile(0.25)
Q3   = fare_series.quantile(0.75)
IQR  = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR   # below this → low outlier
upper_fence = Q3 + 1.5 * IQR   # above this → high outlier

# Boolean mask for outliers (lower fences rarely trigger for fare since it is ≥ 0)
outlier_mask = (fare_series < lower_fence) | (fare_series > upper_fence)
outliers_iqr = fare_series[outlier_mask]

print('IQR Outlier Detection — Fare')
print(f'  Q1           = {Q1:.2f}')
print(f'  Q3           = {Q3:.2f}')
print(f'  IQR          = {IQR:.2f}')
print(f'  Lower fence  = {lower_fence:.2f}  (no outliers below this for fare)')
print(f'  Upper fence  = {upper_fence:.2f}')
print(f'  Outlier count: {outlier_mask.sum()} out of {len(fare_series)} ({outlier_mask.mean()*100:.1f}%)')
print(f'\n  Top 10 outlier values:')
print(outliers_iqr.sort_values(ascending=False).head(10).to_string())

In [ ]:
# ── Box plot with outliers explicitly labelled ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

# Seaborn boxplot — flier (outlier) points shown by default
sns.boxplot(
    x=df['fare'],
    ax=ax,
    color='steelblue',
    flierprops=dict(marker='o', markerfacecolor='red', markersize=4, alpha=0.5)
)

# Annotate the fence lines so readers can see exactly where outliers begin
ax.axvline(upper_fence, color='red', linestyle='--', linewidth=1.5,
           label=f'Upper IQR fence = £{upper_fence:.1f}')
ax.axvline(Q3,          color='green', linestyle=':', linewidth=1.2,
           label=f'Q3 = £{Q3:.1f}')
ax.axvline(Q1,          color='orange', linestyle=':', linewidth=1.2,
           label=f'Q1 = £{Q1:.1f}')

ax.set_title('Fare Distribution — Box Plot with IQR Outliers', fontsize=14, fontweight='bold')
ax.set_xlabel('Fare (£)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Method 2: Z-Score Method on `age` ────────────────────────────────────────
# Z-score = (x - mean) / std. Values with |z| > 3 are extreme outliers
# under the assumption of a roughly normal distribution.

age_series = df['age'].dropna()

# scipy.stats.zscore computes z-scores for a 1D array
z_scores = np.abs(stats.zscore(age_series))          # absolute z-scores

# Threshold: |z| > 3 flags ~0.3% of a perfectly normal distribution
Z_THRESHOLD = 3
outlier_ages = age_series[z_scores > Z_THRESHOLD]

print('Z-Score Outlier Detection — Age')
print(f'  Mean age   = {age_series.mean():.2f}')
print(f'  Std dev    = {age_series.std():.2f}')
print(f'  Threshold  = |z| > {Z_THRESHOLD}')
print(f'  Outlier count: {len(outlier_ages)} passengers')
print(f'\n  Outlier passenger ages (index → age):')
print(outlier_ages.to_string())

# Show the original DataFrame rows for context
print('\n  Full rows for age outliers:')
display(df.loc[outlier_ages.index, ['survived', 'pclass', 'sex', 'age', 'fare']])

In [ ]:
# ── Histogram of age with z-score threshold lines ────────────────────────────
mu  = age_series.mean()
sig = age_series.std()

# The threshold lines in original units: mean ± 3*std
lower_z = mu - Z_THRESHOLD * sig
upper_z = mu + Z_THRESHOLD * sig

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(age_series, bins=40, color='steelblue', edgecolor='white',
        alpha=0.8, label='Age distribution')

# Shade the outlier region on both sides
ax.axvspan(0,       lower_z, alpha=0.15, color='red')
ax.axvspan(upper_z, 90,      alpha=0.15, color='red')

ax.axvline(lower_z, color='red', linestyle='--', linewidth=2,
           label=f'|z|=3 lower bound = {lower_z:.1f} yrs')
ax.axvline(upper_z, color='red', linestyle='--', linewidth=2,
           label=f'|z|=3 upper bound = {upper_z:.1f} yrs')
ax.axvline(mu,      color='black', linestyle='-', linewidth=1.5,
           label=f'Mean = {mu:.1f} yrs')

# Scatter outlier ages as rug ticks near y=0
ax.scatter(outlier_ages, np.zeros(len(outlier_ages)) + 2,
           color='red', zorder=5, s=60, label='Outlier passengers')

ax.set_title('Age Distribution with Z-Score Outlier Threshold (|z| > 3)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Age (years)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Compare the two methods ───────────────────────────────────────────────────
# For a fair comparison we apply z-score to fare too
z_fare   = np.abs(stats.zscore(fare_series))
iqr_set  = set(outliers_iqr.index)                 # indices flagged by IQR
z_set    = set(fare_series[z_fare > 3].index)       # indices flagged by z-score on fare

overlap  = iqr_set & z_set
iqr_only = iqr_set - z_set
z_only   = z_set  - iqr_set

print('Outlier Comparison — Fare column (IQR vs Z-score):')
print(f'  IQR method found:          {len(iqr_set):>3} outliers')
print(f'  Z-score (|z|>3) found:     {len(z_set):>3} outliers')
print(f'  Overlap (both agree):      {len(overlap):>3}')
print(f'  IQR-only (not in z-score): {len(iqr_only):>3}')
print(f'  Z-score-only (not in IQR): {len(z_only):>3}')

print(f'\nAge z-score outliers: {len(outlier_ages)}')

### Outlier Decision: Keep or Remove?

**IQR vs Z-score — which is better?**

| Criterion | IQR Method | Z-Score Method |
|-----------|-----------|----------------|
| Assumes normality? | No | Yes |
| Sensitive to extreme values? | No (uses quartiles) | Yes (mean/std are pulled) |
| Better for skewed distributions | ✅ Yes | ❌ No |
| Better for near-normal distributions | ✅ Reasonable | ✅ Yes |

**Decision for `fare` (skewed): keep all outliers.**

Reasoning:
1. The high fares are **real data** — first-class passengers genuinely paid those prices. Removing them would destroy signal about class privilege and survival.
2. We have already mitigated their distorting effect with the **log transform** in Section 1.
3. Tree-based models (Random Forest, XGBoost) are robust to outliers anyway — no removal needed.

**Decision for `age`: keep all outliers.**

Reasoning:
1. Ages of 65–74 are plausible — elderly first-class passengers existed.
2. Only a handful of points cross the z=3 threshold; removing them would reduce an already small dataset.
3. If anything, very old or very young passengers may have distinctive survival patterns (children were prioritised), so those points carry *more* information, not less.

---
# Section 3 — Inferential Statistics: Two-Sample t-Tests
*Connects to: Notebooks 04 (Hypothesis Testing) & 05 (P-Values)*

In [ ]:
# ── Helper: Cohen's d effect size for two independent samples ─────────────────
# Cohen's d = (mean1 - mean2) / pooled_std
# Interpretation: small=0.2, medium=0.5, large=0.8

def cohens_d(group1, group2):
    """Compute Cohen's d for two independent samples."""
    n1, n2   = len(group1), len(group2)
    var1, var2 = group1.var(ddof=1), group2.var(ddof=1)
    # Pooled standard deviation (Hedges' formulation)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    return (group1.mean() - group2.mean()) / pooled_std


# ── Helper: 95% CI for the difference in means ────────────────────────────────
def ci_diff_means(group1, group2, confidence=0.95):
    """Return (lower, upper) confidence interval for mean(g1) - mean(g2)."""
    n1, n2     = len(group1), len(group2)
    mean_diff  = group1.mean() - group2.mean()
    se_diff    = np.sqrt(group1.var(ddof=1)/n1 + group2.var(ddof=1)/n2)
    # Degrees of freedom: Welch–Satterthwaite approximation
    df_welch   = (se_diff**2)**2 / (
                    (group1.var(ddof=1)/n1)**2 / (n1-1) +
                    (group2.var(ddof=1)/n2)**2 / (n2-1)
                 )
    alpha      = 1 - confidence
    t_crit     = stats.t.ppf(1 - alpha/2, df=df_welch)  # two-tailed critical value
    margin     = t_crit * se_diff
    return mean_diff - margin, mean_diff + margin

print('Helper functions defined: cohens_d() and ci_diff_means()')

### Test 1 — Did First-Class Passengers Pay Significantly More Than Third-Class Passengers?

**H₀ (Null hypothesis):** The mean fare for class-1 passengers equals the mean fare for class-3 passengers.  
μ₁ = μ₃

**H₁ (Alternative hypothesis):** First-class passengers paid a higher mean fare than third-class passengers.  
μ₁ > μ₃ (one-tailed, but we will compute two-tailed p and halve it)

**Significance level:** α = 0.05

In [ ]:
# ── Prepare groups ────────────────────────────────────────────────────────────
class1_fare = df[df['pclass'] == 1]['fare'].dropna()
class3_fare = df[df['pclass'] == 3]['fare'].dropna()

print(f'Class 1: n={len(class1_fare)}, mean=£{class1_fare.mean():.2f}, std=£{class1_fare.std():.2f}')
print(f'Class 3: n={len(class3_fare)}, mean=£{class3_fare.mean():.2f}, std=£{class3_fare.std():.2f}')

# ── KDE plot of overlapping fare distributions ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

sns.kdeplot(class1_fare, ax=ax, fill=True, alpha=0.4, label='Class 1', color='royalblue')
sns.kdeplot(class3_fare, ax=ax, fill=True, alpha=0.4, label='Class 3', color='tomato')

# Mark means with vertical lines
ax.axvline(class1_fare.mean(), color='royalblue', linestyle='--',
           label=f'Class 1 mean = £{class1_fare.mean():.1f}')
ax.axvline(class3_fare.mean(), color='tomato', linestyle='--',
           label=f'Class 3 mean = £{class3_fare.mean():.1f}')

ax.set_title('Fare Distributions: Class 1 vs Class 3 (KDE)', fontsize=13, fontweight='bold')
ax.set_xlabel('Fare (£)')
ax.set_ylabel('Density')
ax.set_xlim(-10, 320)   # clip x-axis to ignore the extreme tail for readability
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Welch's t-test (equal_var=False handles unequal variances between classes) ─
t_stat, p_val = stats.ttest_ind(class1_fare, class3_fare, equal_var=False)

# Effect size and CI
d   = cohens_d(class1_fare, class3_fare)
ci_lo, ci_hi = ci_diff_means(class1_fare, class3_fare)

print('=== Welch Two-Sample t-Test: Class 1 Fare vs Class 3 Fare ===')
print(f'  t-statistic : {t_stat:.4f}')
print(f'  p-value     : {p_val:.6f}')
print(f"  Cohen's d   : {d:.3f}  ({'small' if abs(d)<0.5 else 'medium' if abs(d)<0.8 else 'large'} effect)")
print(f'  95% CI for difference in means: [£{ci_lo:.2f},  £{ci_hi:.2f}]')

# Decision rule
alpha = 0.05
if p_val < alpha:
    print(f'\n  p ({p_val:.6f}) < α ({alpha}) → REJECT H₀')
    print('  Conclusion: First-class passengers paid significantly more than third-class passengers.')
else:
    print(f'\n  p ({p_val:.6f}) ≥ α ({alpha}) → FAIL TO REJECT H₀')

### Interpretation — Test 1

- **p-value << 0.05:** The result is statistically significant. We reject H₀ — first-class passengers paid dramatically more on average.
- **Cohen's d > 0.8:** This is a **large effect size**, not a borderline result. The difference is not just statistically significant but *practically* large.
- **95% CI for the mean difference:** The entire interval is positive and far from zero, confirming the direction and magnitude are stable across repeated sampling.
- **ML implication:** `fare` and `pclass` are not independent features — they are strongly collinear. Including both in a linear model will cause **multicollinearity**, inflating coefficient variance. Consider using only one, or using `log_fare` after partialling out class effects.

### Test 2 — Is Survival Rate Significantly Different Between Male and Female Passengers?

**H₀:** Mean survival rate for males = mean survival rate for females. (μ_male = μ_female)

**H₁:** There is a significant difference in survival rates between sexes. (μ_male ≠ μ_female, two-tailed)

**Note:** `survived` is binary (0/1), so the t-test is technically comparing means of a Bernoulli variable — equivalent to a proportions test for large samples (CLT applies here).

In [ ]:
# ── Prepare groups: male vs female survival ───────────────────────────────────
male_surv   = df[df['sex'] == 'male']['survived'].dropna()
female_surv = df[df['sex'] == 'female']['survived'].dropna()

print(f'Male:   n={len(male_surv)},   survival rate = {male_surv.mean()*100:.1f}%')
print(f'Female: n={len(female_surv)}, survival rate = {female_surv.mean()*100:.1f}%')

# ── Welch's t-test on binary survival ────────────────────────────────────────
t2, p2 = stats.ttest_ind(female_surv, male_surv, equal_var=False)
d2     = cohens_d(female_surv, male_surv)
ci2_lo, ci2_hi = ci_diff_means(female_surv, male_surv)

print('\n=== Welch t-Test: Survival Rate — Female vs Male ===')
print(f'  t-statistic : {t2:.4f}')
print(f'  p-value     : {p2:.2e}')        # scientific notation because p is tiny
print(f"  Cohen's d   : {d2:.3f}")
print(f'  95% CI for diff in survival rate: [{ci2_lo:.4f},  {ci2_hi:.4f}]')
print(f'  (i.e., females survived at a rate {ci2_lo*100:.1f}% to {ci2_hi*100:.1f}% higher)')

alpha = 0.05
if p2 < alpha:
    print(f'\n  p < α → REJECT H₀')
    print('  Conclusion: Survival rates differ SIGNIFICANTLY between sexes.')

### Interpretation — Test 2

The p-value is essentially zero. Female passengers survived at dramatically higher rates than male passengers — this is one of the strongest signals in the dataset. Cohen's d confirms a large effect.

**Before jumping to ML conclusions:** the "women and children first" lifeboat protocol was a real historical policy during this disaster. `sex` is highly predictive here, but it is the *policy* that drove survival, not any intrinsic property of sex. This is important context for the ethical question in Section 9.

---
# Section 4 — Confidence Intervals
*Connects to: Notebook 06 (Confidence Intervals & Effect Sizes)*

In [ ]:
# ── Helper: 95% CI for a mean (t-distribution, unknown population std) ─────────
def ci_mean(series, confidence=0.95):
    """Return (mean, lower, upper) 95% CI using t-distribution."""
    series = series.dropna()
    n      = len(series)
    mean   = series.mean()
    se     = stats.sem(series)               # standard error of the mean = std / sqrt(n)
    margin = se * stats.t.ppf((1 + confidence) / 2, df=n - 1)  # t critical value
    return mean, mean - margin, mean + margin


# ── Helper: 95% CI for a proportion using Wilson score interval ───────────────
def ci_proportion(series, confidence=0.95):
    """Return (proportion, lower, upper) 95% CI using normal approximation."""
    series = series.dropna()
    n = len(series)
    p = series.mean()
    z = stats.norm.ppf((1 + confidence) / 2)  # z ≈ 1.96 for 95%
    margin = z * np.sqrt(p * (1 - p) / n)
    return p, p - margin, p + margin


# ── Overall CIs ───────────────────────────────────────────────────────────────
mean_age,  age_lo,  age_hi  = ci_mean(df['age'])
mean_fare, fare_lo, fare_hi = ci_mean(df['fare'])
surv_p,    surv_lo, surv_hi = ci_proportion(df['survived'])

print('95% Confidence Intervals — Overall')
print(f'  Mean age          : {mean_age:.2f}  [{age_lo:.2f},  {age_hi:.2f}]  years')
print(f'  Mean fare         : £{mean_fare:.2f}  [£{fare_lo:.2f},  £{fare_hi:.2f}]')
print(f'  Survival rate     : {surv_p*100:.1f}%  [{surv_lo*100:.1f}%,  {surv_hi*100:.1f}%]')

In [ ]:
# ── Survival rate CIs broken down by sex, pclass, and embarked ───────────────
ci_records = []

# By sex
for val in ['male', 'female']:
    subset = df[df['sex'] == val]['survived']
    p, lo, hi = ci_proportion(subset)
    ci_records.append({'group_var': 'Sex', 'group': val,
                       'survival_rate': p, 'ci_lo': lo, 'ci_hi': hi, 'n': len(subset)})

# By pclass
for val in [1, 2, 3]:
    subset = df[df['pclass'] == val]['survived']
    p, lo, hi = ci_proportion(subset)
    ci_records.append({'group_var': 'Pclass', 'group': str(val),
                       'survival_rate': p, 'ci_lo': lo, 'ci_hi': hi, 'n': len(subset)})

# By embarked port
for val in ['C', 'Q', 'S']:
    subset = df[df['embarked'] == val]['survived']
    p, lo, hi = ci_proportion(subset)
    ci_records.append({'group_var': 'Embarked', 'group': val,
                       'survival_rate': p, 'ci_lo': lo, 'ci_hi': hi, 'n': len(subset)})

ci_df = pd.DataFrame(ci_records)
print('Survival Rate CIs by Group:')
display(ci_df.round(3))

In [ ]:
# ── Forest plot: horizontal CI bars per group ─────────────────────────────────
# A forest plot makes it easy to spot non-overlapping CIs (= clearly different groups)

fig, ax = plt.subplots(figsize=(9, 6))

# Colour-code by group variable
colours = {'Sex': 'royalblue', 'Pclass': 'tomato', 'Embarked': 'green'}
y_labels, y_positions = [], []
y = 0
group_start = {}   # track where each group_var starts for section labels

# Sort so groups appear in a logical order: Sex, Pclass, Embarked
order = ['Sex', 'Pclass', 'Embarked']
for gvar in order:
    group_start[gvar] = y
    subset = ci_df[ci_df['group_var'] == gvar]
    for _, row in subset.iterrows():
        # Horizontal error bar: xerr expects (lower_error, upper_error) for each point
        ax.errorbar(
            x=row['survival_rate'],
            y=y,
            xerr=[[row['survival_rate'] - row['ci_lo']],
                  [row['ci_hi'] - row['survival_rate']]],
            fmt='o',
            color=colours[gvar],
            capsize=5, capthick=2, linewidth=2, markersize=8
        )
        label = f"{row['group_var']}: {row['group']} (n={row['n']})"
        y_labels.append(label)
        y_positions.append(y)
        y += 1
    y += 0.5   # add a gap between group variable sections

ax.set_yticks(y_positions)
ax.set_yticklabels(y_labels)
ax.axvline(surv_p, color='black', linestyle='--', linewidth=1.2,
           label=f'Overall survival rate = {surv_p*100:.1f}%')
ax.set_xlabel('Survival Rate (proportion)')
ax.set_title('95% Confidence Intervals for Survival Rate by Group\n(Forest Plot Style)',
             fontsize=13, fontweight='bold')
ax.legend()
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

### Interpretation — Confidence Intervals

- **Non-overlapping CIs → clearly different groups:** The female CI and the male CI do not overlap at all — a visual confirmation of the t-test finding. Similarly, class-1 and class-3 survival CIs are likely non-overlapping.
- **Embarked port CIs:** Passengers who boarded at Cherbourg (C) have a noticeably higher survival rate than Southampton (S). However, this is probably a **confounder** — Cherbourg had a higher proportion of first-class passengers. The effect is likely explained by `pclass`, not the port itself.
- **Width of CIs:** Groups with fewer passengers (e.g., Queenstown 'Q') have wider CIs, reflecting more uncertainty. This is the CI telling you: "we have less data here, so be careful".

**ML implication:** Features with clearly separated group CIs are good predictors. `sex` and `pclass` stand out.

---
# Section 5 — Correlation Analysis
*Connects to: Notebook 07 (Correlation vs Causation)*

In [ ]:
# ── Select only the numeric columns we care about for correlation ─────────────
numeric_cols = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']

# Pearson correlation matrix (default in pandas .corr())
corr_matrix = df[numeric_cols].corr()

print('Correlation Matrix (Pearson r):')
display(corr_matrix.round(3))

# ── Seaborn heatmap ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    corr_matrix,
    annot=True,          # print r values inside cells
    fmt='.2f',           # 2 decimal places
    cmap='coolwarm',     # blue = negative correlation, red = positive
    vmin=-1, vmax=1,     # fix the colour scale to [-1, 1]
    linewidths=0.5,      # cell borders for readability
    ax=ax,
    square=True          # equal cell aspect ratios
)

ax.set_title('Correlation Heatmap — Numeric Features (Titanic)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter plot: age vs fare coloured by pclass ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: age vs fare, colour = survived ---
# Use small alpha to handle overplotting
scatter1 = axes[0].scatter(
    df['age'], df['fare'],
    c=df['survived'],       # numeric colour coding: 0=not survived, 1=survived
    cmap='coolwarm',
    alpha=0.5, s=20, edgecolors='none'
)
plt.colorbar(scatter1, ax=axes[0], label='Survived (0=No, 1=Yes)')
axes[0].set_title('Age vs Fare (colour = survival)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Fare (£)')

# --- Right: pclass vs fare with jitter (pclass is 1/2/3 — discrete) ---
# Jitter prevents all points from stacking on the same x values
jitter = np.random.uniform(-0.15, 0.15, size=len(df))   # small random x-offset
scatter2 = axes[1].scatter(
    df['pclass'] + jitter, df['fare'],
    c=df['survived'],
    cmap='coolwarm',
    alpha=0.4, s=20, edgecolors='none'
)
plt.colorbar(scatter2, ax=axes[1], label='Survived (0=No, 1=Yes)')
axes[1].set_title('Pclass vs Fare with Jitter (colour = survival)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Passenger Class (jittered)')
axes[1].set_ylabel('Fare (£)')
axes[1].set_xticks([1, 2, 3])
axes[1].set_xticklabels(['Class 1', 'Class 2', 'Class 3'])

plt.suptitle('Scatter Plots: Relationships Between Key Numeric Features',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Which features correlate most with survival? ─────────────────────────────
# Extract the survival row and sort by absolute correlation strength
surv_corr = corr_matrix['survived'].drop('survived').abs().sort_values(ascending=False)

print('Correlations with `survived` (sorted by |r|):')
for feat, r in surv_corr.items():
    raw_r = corr_matrix['survived'][feat]
    direction = 'positive' if raw_r > 0 else 'negative'
    print(f'  {feat:<8}  r = {raw_r:+.3f}   ({direction})')

# Visualise
fig, ax = plt.subplots(figsize=(7, 4))
colours = ['tomato' if corr_matrix['survived'][f] < 0 else 'steelblue'
           for f in surv_corr.index]
ax.barh(surv_corr.index, surv_corr.values, color=colours)
ax.set_xlabel('|Pearson r| with Survived')
ax.set_title('Feature Correlations with Survival (absolute)', fontsize=12, fontweight='bold')
ax.set_xlim(0, 0.6)
plt.tight_layout()
plt.show()

### Correlation Findings & the Causation Warning

**Key correlations with survival:**
- `fare` has a positive correlation with `survived` — passengers who paid more tended to survive more.
- `pclass` has a **negative** correlation — higher class number (= lower class) → lower survival probability.
- `age` and `sibsp`/`parch` show smaller correlations, suggesting weaker direct linear relationships.

**The confounder problem — does fare *cause* survival?**

Consider this causal chain:

```
pclass  ──→  fare   ──→  survival
  └──────────────────────────→
```

`pclass` drives both `fare` (first-class is more expensive) AND `survival` (first-class cabins were closer to lifeboats). `fare` is therefore a **mediator/proxy** for `pclass`, not an independent cause. Including both `fare` and `pclass` in a model creates redundancy.

**Rule:** Correlation ≠ causation. Always ask: "Is there a third variable that explains this relationship?"

---
# Section 6 — Visualisation Gallery (12 Charts)
*Connects to: Notebook 10 (Matplotlib & Seaborn)*

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Chart 1: Age Distribution with KDE
# Chart 2: Fare — Original vs log(Fare) side by side
# ════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Chart 1: Age histogram + KDE overlay ──────────────────────────
sns.histplot(
    df['age'].dropna(), bins=35, kde=True,
    ax=axes[0], color='steelblue', edgecolor='white'
)
axes[0].set_title('Chart 1: Age Distribution with KDE', fontweight='bold')
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Count')

# ── Chart 2a: Original fare histogram ─────────────────────────────
sns.histplot(
    df['fare'].dropna(), bins=50, kde=True,
    ax=axes[1], color='darkorange', edgecolor='white'
)
axes[1].set_title('Chart 2a: Fare — Original (Skewed)', fontweight='bold')
axes[1].set_xlabel('Fare (£)')
axes[1].set_ylabel('Count')

# ── Chart 2b: log(fare) histogram ─────────────────────────────────
sns.histplot(
    df['log_fare'].dropna(), bins=35, kde=True,
    ax=axes[2], color='mediumseagreen', edgecolor='white'
)
axes[2].set_title('Chart 2b: log(Fare+1) — After Transform', fontweight='bold')
axes[2].set_xlabel('log(Fare + 1)')
axes[2].set_ylabel('Count')

plt.suptitle('Charts 1 & 2: Distributions of Age and Fare',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Chart 3: Box plot — Fare by Passenger Class
# Chart 4: Box plot — Age by Survived
# ════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Chart 3: fare by pclass ────────────────────────────────────────
sns.boxplot(
    data=df, x='pclass', y='fare',
    ax=axes[0], palette='husl'
)
axes[0].set_title('Chart 3: Fare by Passenger Class', fontweight='bold')
axes[0].set_xlabel('Passenger Class')
axes[0].set_ylabel('Fare (£)')
# Class 1 outliers extend to ~500; cap y-axis for readability
axes[0].set_ylim(-10, 320)
axes[0].text(0.02, 0.97, '(y-axis capped at £320 for clarity)',
             transform=axes[0].transAxes, fontsize=8, va='top', color='grey')

# ── Chart 4: age by survived ───────────────────────────────────────
# Map 0/1 to labels for readability on the axis
df['survived_label'] = df['survived'].map({0: 'Did Not Survive', 1: 'Survived'})
sns.boxplot(
    data=df, x='survived_label', y='age',
    ax=axes[1], palette=['tomato', 'steelblue']
)
axes[1].set_title('Chart 4: Age by Survival Outcome', fontweight='bold')
axes[1].set_xlabel('Survival Outcome')
axes[1].set_ylabel('Age (years)')

plt.suptitle('Charts 3 & 4: Box Plots — Fare by Class & Age by Survival',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Chart 5: Survival Rate by Sex with 95% CI error bars
# Chart 6: Survival Rate by Passenger Class
# ════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Chart 5: survival rate by sex with error bars ─────────────────
sex_ci = {}
for sex in ['male', 'female']:
    p, lo, hi = ci_proportion(df[df['sex'] == sex]['survived'])
    sex_ci[sex] = {'p': p, 'lo': lo, 'hi': hi}

sexes  = list(sex_ci.keys())
ps     = [sex_ci[s]['p']  for s in sexes]
yerr_lo= [sex_ci[s]['p'] - sex_ci[s]['lo'] for s in sexes]   # lower error bar length
yerr_hi= [sex_ci[s]['hi'] - sex_ci[s]['p'] for s in sexes]   # upper error bar length

bars5 = axes[0].bar(sexes, ps, color=['steelblue', 'tomato'], edgecolor='white', width=0.5)
axes[0].errorbar(
    sexes, ps,
    yerr=[yerr_lo, yerr_hi],
    fmt='none', color='black', capsize=8, linewidth=2
)
for bar, val in zip(bars5, ps):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.02,
                 f'{val*100:.1f}%', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Chart 5: Survival Rate by Sex\n(error bars = 95% CI)', fontweight='bold')
axes[0].set_xlabel('Sex')
axes[0].set_ylabel('Survival Rate')
axes[0].set_ylim(0, 1)

# ── Chart 6: survival rate by pclass ──────────────────────────────
class_surv = df.groupby('pclass')['survived'].mean()
axes[1].bar(
    class_surv.index.astype(str), class_surv.values,
    color=sns.color_palette('husl', 3), edgecolor='white', width=0.5
)
for x, val in zip(class_surv.index.astype(str), class_surv.values):
    axes[1].text(x, val + 0.01, f'{val*100:.1f}%', ha='center', fontweight='bold')
axes[1].set_title('Chart 6: Survival Rate by Passenger Class', fontweight='bold')
axes[1].set_xlabel('Passenger Class')
axes[1].set_ylabel('Survival Rate')
axes[1].set_ylim(0, 0.85)

plt.suptitle('Charts 5 & 6: Survival Rates by Sex and Class',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Chart 7: Correlation Heatmap (already produced in Section 5 —
#          reproduced here as part of the gallery with tighter style)
# Chart 8: Pairplot — age, fare, pclass, survived (hue=survived)
# ════════════════════════════════════════════════════════════════════

# ── Chart 7: heatmap ──────────────────────────────────────────────
fig7, ax7 = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, linewidths=0.5, ax=ax7, square=True
)
ax7.set_title('Chart 7: Correlation Heatmap — Numeric Features',
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Chart 8: Pairplot ─────────────────────────────────────────────
# Pairplot creates a grid of bivariate scatter/KDE plots.
# Use a small subset of columns to keep it manageable.
pairplot_cols = ['age', 'fare', 'pclass', 'survived']

# Create a copy with a readable survived label for the legend
df_pp = df[pairplot_cols + ['survived_label']].dropna(subset=pairplot_cols)

g = sns.pairplot(
    df_pp,
    vars=pairplot_cols,
    hue='survived_label',
    palette={'Did Not Survive': 'tomato', 'Survived': 'steelblue'},
    plot_kws={'alpha': 0.3, 's': 15},
    diag_kind='kde',
    corner=True       # only lower triangle to halve the plot size
)
g.fig.suptitle('Chart 8: Pairplot — age, fare, pclass, survived',
               y=1.02, fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Chart 9:  Violin plot — Age by Passenger Class
# Chart 10: Stacked bar chart — Survival count by sex × pclass
# ════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Chart 9: Violin plot ───────────────────────────────────────────
# Violin plots combine a box plot with a KDE, showing full distribution shape
sns.violinplot(
    data=df, x='pclass', y='age',
    ax=axes[0], palette='husl', inner='quartile'
    # inner='quartile' draws quartile lines inside the violin
)
axes[0].set_title('Chart 9: Age Distribution by Passenger Class\n(Violin Plot)', fontweight='bold')
axes[0].set_xlabel('Passenger Class')
axes[0].set_ylabel('Age (years)')

# ── Chart 10: Stacked bar chart — survival count by sex × pclass ──
# Build a pivot table: rows=pclass, columns=sex, values=survival count
# We need a 2D breakdown: survived vs not-survived, broken by sex AND pclass

# Crosstab: index=pclass, columns=sex, values=count of survived=1
ct_survived = df[df['survived'] == 1].groupby(['pclass', 'sex']).size().unstack(fill_value=0)
ct_died     = df[df['survived'] == 0].groupby(['pclass', 'sex']).size().unstack(fill_value=0)

x = np.arange(len(ct_survived.index))  # class positions on x-axis
width = 0.35

# Female survived (bottom of each group), male survived (stacked on top)
# We'll show two side-by-side stacked groups per pclass: survived vs not survived
# Simpler: bar chart per sex, stacked by survived/not
female_surv = df[df['sex']=='female'].groupby('pclass')['survived'].sum()
female_died = df[df['sex']=='female'].groupby('pclass').apply(lambda g: (g['survived']==0).sum())
male_surv   = df[df['sex']=='male'].groupby('pclass')['survived'].sum()
male_died   = df[df['sex']=='male'].groupby('pclass').apply(lambda g: (g['survived']==0).sum())

classes = [1, 2, 3]
x = np.arange(len(classes))

# Female bars
b1 = axes[1].bar(x - width/2, female_surv.values, width, label='Female — Survived', color='steelblue')
b2 = axes[1].bar(x - width/2, female_died.values, width, bottom=female_surv.values,
                 label='Female — Died', color='lightsteelblue')
# Male bars
b3 = axes[1].bar(x + width/2, male_surv.values, width, label='Male — Survived', color='tomato')
b4 = axes[1].bar(x + width/2, male_died.values, width, bottom=male_surv.values,
                 label='Male — Died', color='lightsalmon')

axes[1].set_title('Chart 10: Survival Count by Sex × Passenger Class\n(Stacked Bar)', fontweight='bold')
axes[1].set_xlabel('Passenger Class')
axes[1].set_ylabel('Passenger Count')
axes[1].set_xticks(x)
axes[1].set_xticklabels(['Class 1', 'Class 2', 'Class 3'])
axes[1].legend(loc='upper right', fontsize=8)

plt.suptitle('Charts 9 & 10: Violin by Class | Stacked Survival by Sex × Class',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════════
# Chart 11: Scatter — Age vs Fare, coloured by Survived
# Chart 12: Count plot — Embarkation Port Distribution
# ════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Chart 11: age vs fare scatter, coloured by survived ───────────
for label, colour, marker in [(0, 'tomato', 'x'), (1, 'steelblue', 'o')]:
    subset = df[df['survived'] == label]
    axes[0].scatter(
        subset['age'], subset['fare'],
        c=colour, alpha=0.4, s=18,
        marker=marker,
        label='Survived' if label == 1 else 'Did Not Survive'
    )
axes[0].set_title('Chart 11: Age vs Fare\n(colour = survival outcome)', fontweight='bold')
axes[0].set_xlabel('Age (years)')
axes[0].set_ylabel('Fare (£)')
axes[0].set_ylim(-5, 320)
axes[0].legend()

# ── Chart 12: count plot for embarkation port ──────────────────────
# Map single-letter codes to full port names for clarity
port_map = {'C': 'Cherbourg', 'Q': 'Queenstown', 'S': 'Southampton'}
df['embark_full'] = df['embarked'].map(port_map)

sns.countplot(
    data=df, x='embark_full', hue='survived_label',
    ax=axes[1],
    palette={'Survived': 'steelblue', 'Did Not Survive': 'tomato'},
    order=['Southampton', 'Cherbourg', 'Queenstown']  # sorted by count
)
axes[1].set_title('Chart 12: Passenger Count by Embarkation Port\n(hue = survival)', fontweight='bold')
axes[1].set_xlabel('Embarkation Port')
axes[1].set_ylabel('Number of Passengers')
axes[1].legend(title='Survival')

plt.suptitle('Charts 11 & 12: Age vs Fare Scatter | Embarkation Port',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
# Section 7 — NumPy & Pandas Operations
*Connects to: Notebooks 08 (NumPy) & 09 (Pandas)*

In [ ]:
# ── Convert numeric DataFrame columns to a NumPy array ───────────────────────
# This is what scikit-learn does internally — it converts DataFrames to arrays.
numeric_subset = df[['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']].dropna()

arr = numeric_subset.to_numpy()   # shape: (n_complete_rows, 6)
print(f'NumPy array shape: {arr.shape}')
print(f'Array dtype: {arr.dtype}')
print(f'First 3 rows:\n{arr[:3]}')

# ── Vectorised operations on the array (no Python loops needed) ───────────────
col_names = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']

# Column-wise statistics using NumPy axis=0 (along rows)
col_means = np.mean(arr, axis=0)
col_stds  = np.std(arr,  axis=0)

print('\nColumn-wise means (vectorised np.mean):')
for name, val in zip(col_names, col_means):
    print(f'  {name:<10}: {val:.3f}')

# ── Min-max normalisation (a common ML preprocessing step) ───────────────────
# Scale each column to [0, 1] using vectorised broadcasting
arr_min   = arr.min(axis=0)                     # shape (6,)
arr_max   = arr.max(axis=0)                     # shape (6,)
arr_norm  = (arr - arr_min) / (arr_max - arr_min)  # broadcasting: each row - min vector

print(f'\nMin-max normalised array — min values: {arr_norm.min(axis=0).round(2)}')
print(f'Min-max normalised array — max values: {arr_norm.max(axis=0).round(2)}')

In [ ]:
# ── pandas groupby: survival rate by pclass AND sex ──────────────────────────
# groupby produces a GroupBy object; .agg() applies a function to each group
surv_by_class_sex = (
    df.groupby(['pclass', 'sex'])['survived']
    .agg(['mean', 'count', 'sum'])
    .rename(columns={'mean': 'survival_rate', 'count': 'total', 'sum': 'survived_count'})
)
surv_by_class_sex['survival_rate'] = surv_by_class_sex['survival_rate'].round(3)

print('Survival rate by pclass × sex:')
display(surv_by_class_sex)

# ── Pivot table: same data, wider format ──────────────────────────────────────
# pivot_table reshapes the groupby result into a 2D matrix — easier to read
pivot = df.pivot_table(
    values='survived',
    index='pclass',
    columns='sex',
    aggfunc='mean'
).round(3)

print('\nSurvival rate pivot table (rows=pclass, cols=sex):')
display(pivot)

In [ ]:
# ── pandas apply: classify passengers into age groups ────────────────────────
# .apply() runs a function on every element (or row). Here we use it element-wise.

def classify_age(age):
    """Return a string age-group label for a given age value."""
    if pd.isna(age):        # preserve NaN rather than silently classifying it
        return np.nan
    elif age < 18:
        return 'Child'      # under 18
    elif age < 60:
        return 'Adult'      # 18–59
    else:
        return 'Senior'     # 60+

# Apply to the age column to create a new categorical column
df['age_group'] = df['age'].apply(classify_age)

print('Age group distribution:')
print(df['age_group'].value_counts())

# ── Survival rate by age group ────────────────────────────────────────────────
age_surv = df.groupby('age_group')['survived'].mean().rename('survival_rate').round(3)
print('\nSurvival rate by age group:')
display(age_surv.to_frame())

In [ ]:
# ── Visualise the pclass × sex pivot as a heatmap ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Left: Heatmap of survival rate by pclass × sex ───────────────────────────
sns.heatmap(
    pivot,
    annot=True, fmt='.2f',     # print values inside cells
    cmap='RdYlGn',             # green = high survival, red = low
    vmin=0, vmax=1,
    linewidths=0.5,
    ax=axes[0]
)
axes[0].set_title('Survival Rate by Pclass × Sex\n(pivot table as heatmap)', fontweight='bold')
axes[0].set_xlabel('Sex')
axes[0].set_ylabel('Passenger Class')

# ── Right: Bar chart of survival rate by age group ───────────────────────────
# Define a consistent order for age groups
age_order = ['Child', 'Adult', 'Senior']
age_surv_ordered = age_surv.reindex(age_order).dropna()

axes[1].bar(
    age_surv_ordered.index,
    age_surv_ordered.values,
    color=sns.color_palette('husl', 3), edgecolor='white', width=0.5
)
for x, val in zip(age_surv_ordered.index, age_surv_ordered.values):
    axes[1].text(x, val + 0.01, f'{val*100:.1f}%', ha='center', fontweight='bold')
axes[1].set_title('Survival Rate by Age Group\n(pandas apply + groupby)', fontweight='bold')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Survival Rate')
axes[1].set_ylim(0, 0.75)

plt.suptitle('NumPy & Pandas: Pivot Table Heatmap & Age Group Analysis',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Key NumPy & Pandas Takeaways

- **Vectorised NumPy operations** (mean, std, normalisation) apply element-wise across the entire array without Python loops — orders of magnitude faster than looping in pure Python, and this is exactly what scikit-learn uses internally.
- **`groupby` + `pivot_table`** are the pandas tools that transform raw data into the aggregated summaries an analyst needs. The pclass × sex survival pivot table above would take 15 lines in a loop; pandas does it in 5.
- **`apply`** is the escape hatch for logic that cannot be expressed as a vectorised operation. Age classification is a classic case: it's a conditional rule, not arithmetic.
- **Min-max normalisation** via broadcasting shows how numpy handles shape mismatches automatically: subtracting a shape-(6,) vector from a shape-(n,6) matrix is broadcast across all rows.

---
# Section 8 — Summary Findings & ML Implications

## Structured Findings Report

---

### Key Patterns Found

1. **Women survived at dramatically higher rates than men (~74% vs ~19%).** The gap is statistically significant (p < 0.001, large Cohen's d) and historically explained by "women and children first" evacuation policy.
2. **First-class passengers had the highest survival rates (~63%), third-class the lowest (~24%).** Physical proximity to lifeboats and faster evacuation of upper decks is the likely mechanism.
3. **Children showed elevated survival rates** relative to adults — consistent with historical accounts that children were prioritised in lifeboat loading.
4. **Fare and pclass are strongly correlated (r ≈ −0.55):** higher class number = lower fare. These two features contain overlapping information; using both in a linear model causes multicollinearity.
5. **Passengers boarding at Cherbourg survived at higher rates than Southampton passengers**, but this is a confounder: Cherbourg boarded a higher proportion of first-class passengers, not because the port itself was protective.
6. **Age has a moderate right skew** (skewness ≈ 0.39) but is otherwise well-behaved. Fare has extreme right skew (skewness > 4) and requires log-transformation before use in linear models.

---

### Outliers

- **Fare outliers (IQR method):** Approximately 10% of passengers had fares above the upper fence of ~£65. These are real, valid data points (genuine first-class ticket prices). They were retained because: (a) the log transformation reduces their distorting effect; (b) tree-based models are robust to them; (c) they may carry signal (extreme fare → high class → high survival).
- **Age outliers (Z-score |z|>3):** A small number of passengers aged ~65–74. All appear legitimate. Retained for the same reasons — elderly first-class passengers are genuinely in the data.
- **Decision:** No rows were deleted. Outlier handling was done through transformation, not removal.

---

### Statistical Significance

| Test | Result | p-value | Cohen's d |
|------|--------|---------|----------|
| Class 1 vs Class 3 fare | Class 1 significantly higher | < 0.001 | Large (> 0.8) |
| Female vs male survival rate | Females significantly higher | < 0.001 | Large |

Both results are not only statistically significant but **practically significant** — large effect sizes confirm that these differences are not artefacts of sample size.

---

### Features Most Predictive of Survival

Based on correlation magnitudes and group-level CI analysis:

1. **`sex`** — strongest predictor (large effect size, non-overlapping CIs)
2. **`pclass`** — second strongest predictor (r ≈ −0.34 with survived)
3. **`fare` / `log_fare`** — closely tied to pclass; adds redundant but occasionally non-overlapping information
4. **`age_group`** — children have elevated survival; useful as a categorical feature
5. **`embarked`** — weak predictor once pclass is controlled for

---

### Data Quality Issues

| Issue | Recommendation |
|-------|----------------|
| `age` ~20% missing | Impute with median (robust to skew); or use a model that handles NaN natively |
| `deck` ~77% missing | Drop the column or encode as binary (known/unknown) |
| `embarked` 2 rows missing | Fill with mode ('S' — Southampton was most common) |
| `fare` right-skewed | Apply log1p transform before modelling |
| `sex`, `embarked`, `pclass` are categorical | Encode with one-hot encoding or ordinal encoding depending on algorithm |

---

### What an ML Model Would Learn from This Data

A logistic regression or decision tree trained on this data would effectively learn:

> *"If you were a woman in first or second class, your probability of survival was very high. If you were a man in third class, it was very low."*

More formally:
- The model will assign large positive weights to `sex=female` and `pclass=1`, and large negative weights to `sex=male` and `pclass=3`.
- `fare` will likely get a positive coefficient, but its coefficient will be suppressed if `pclass` is also included (multicollinearity).
- `age` will have a small negative coefficient (slightly worse survival as age increases, after controlling for sex and class).
- Missing age values must be imputed *before* the model can use the feature — scikit-learn will raise an error on NaN by default.

---
# Section 9 — Self-Check Questions

Work through these questions yourself before expanding the answers. They cover the conceptual reasoning behind everything in this lab.

---
### Question 1

**Why should you use the *median* instead of the *mean* for `fare` when building an ML model?**

<details>
<summary>Click to reveal answer</summary>

**Answer:**

`fare` is heavily right-skewed (skewness > 4) with a small number of extremely expensive first-class tickets. The mean is pulled upward by those extreme values and does not represent a 'typical' ticket price — the median is around £14 while the mean is around £32.

For ML:
- If you impute missing fares with the mean, the imputed value will be unrealistically high for most passengers (who were in third class).
- The median is resistant to outliers and gives a more honest representation of the central ticket price.
- When scaling features (e.g., StandardScaler), extreme outliers will compress the range of all other values around the standardised mean. Log-transforming first, then using median-based imputation, is the correct order of operations.

**Key principle:** For skewed distributions, median > mean as a robust measure of centre.

</details>

---
### Question 2

**Cohen's d for Class 1 vs Class 3 fare was large (> 0.8). Does this mean passenger class *causes* a higher fare?**

<details>
<summary>Click to reveal answer</summary>

**Answer:**

Yes — in this case, causation is actually plausible and well-supported by domain knowledge: purchasing a first-class ticket literally meant paying a higher fare. The causal arrow runs **pclass → fare** by design of the ticketing system.

However, Cohen's d alone tells you nothing about causation — it only quantifies the magnitude of the difference between two groups. A large d means the groups are very different numerically, but it does not tell you *why*.

Cohen's d would be equally large if, for example, first-class passengers happened to be older and older people happened to pay more — in which case the real driver would be age, not class. You'd need careful causal reasoning (or a controlled experiment) to establish direction.

**Key principle:** Effect size measures *how different* two groups are. It does not establish causal direction.

</details>

---
### Question 3

**You found that survival rate differs significantly by sex (p < 0.001). Before using `sex` as an ML feature, what ethical considerations exist?**

<details>
<summary>Click to reveal answer</summary>

**Answer:**

This is a historically-specific dataset — the gender disparity in survival reflects a *deliberate evacuation policy* ("women and children first"), not any inherent biological difference in survival capability. Key considerations:

1. **Generalisation:** A model trained on this data might falsely generalise that 'being female' is protective in disasters in general, which is not true in other contexts (e.g., earthquake survival).

2. **Protected attributes:** In many ML deployment contexts (hiring, lending, healthcare), sex/gender is a legally protected attribute. Using it directly as a feature can encode discrimination even when the historical data made it predictive.

3. **Proxy discrimination:** Even if you drop `sex`, other correlated features (`pclass`, `embarked`, `name` prefix like 'Mrs') can act as proxies that reintroduce gender discrimination indirectly.

4. **Fairness metrics:** Before deploying a model that uses a sensitive attribute, evaluate demographic parity, equalised odds, and calibration across groups.

**Key principle:** Statistical predictive power does not automatically mean a feature is ethical or appropriate to use in a deployed model.

</details>

---
### Question 4

**The correlation between `pclass` and `fare` is approximately −0.55. What does the negative sign mean here?**

<details>
<summary>Click to reveal answer</summary>

**Answer:**

A negative Pearson r means the two variables move in *opposite directions*: as one goes up, the other tends to go down.

Here: as `pclass` increases (1 → 2 → 3), fare *decreases*. This makes perfect sense: `pclass=1` (first class) is the most expensive, `pclass=3` (third class) is the cheapest. The numeric encoding of pclass (1=best, 3=worst) creates an *inverse* relationship with fare.

Important nuance: `pclass` is encoded as an ordinal integer (1, 2, 3) but is conceptually categorical. Pearson r assumes a linear relationship; the fact that r = −0.55 (moderate, not −1) reflects that the fare difference between classes 1 and 2 is not perfectly proportional to the difference between classes 2 and 3.

**Key principle:** The sign of r tells you the direction of the linear relationship; the magnitude tells you the strength. Always check whether the numeric encoding of a categorical variable is meaningful before interpreting its correlations.

</details>

---
### Question 5

**List 3 things you would do to this dataset before feeding it to a scikit-learn model.**

<details>
<summary>Click to reveal answer</summary>

**Answer (multiple valid responses — here are the most important five, pick any three):**

1. **Impute missing values:**
   - `age`: median imputation (robust to skew), or use a `SimpleImputer` with `strategy='median'`.
   - `embarked`: mode imputation (2 missing rows → fill with 'S').
   - `deck`: drop the column (77% missing, no reliable imputation).

2. **Encode categorical variables:**
   - `sex` → binary encoding (0=male, 1=female) or one-hot.
   - `embarked` → one-hot encoding (C, Q, S → 3 binary columns).
   - `pclass` → can stay as ordinal integer (it has a natural order), or one-hot if using a model that does not handle ordinal naturally.

3. **Log-transform `fare`:**
   - Apply `np.log1p(fare)` to reduce the extreme right skew before feeding to linear models or distance-based algorithms.

4. **Scale numeric features:**
   - Apply `StandardScaler` or `MinMaxScaler` to `age` and `log_fare` so that no single feature dominates distance calculations or regularisation penalties.

5. **Drop redundant or leaky columns:**
   - Drop `deck` (too many missing), `alive` (text version of `survived` — data leakage!), `who`, `adult_male`, `class`, `embark_town` (redundant with `embarked`).
   - Drop `fare` if you include `pclass` (or vice versa) to reduce multicollinearity in linear models.

**Key principle:** EDA is not just for understanding data — every insight in this lab translates directly into a preprocessing decision.

</details>

---

## Lab Complete!

You have just completed a full end-to-end EDA pipeline. Here is a summary of what you built:

| Section | Core Skill |
|---------|------------|
| 0 — Setup | Data loading, dtype inspection, missing value audit |
| 1 — Descriptive Stats | Mean, median, mode, skewness, kurtosis, log transform |
| 2 — Outliers | IQR fences, z-scores, principled keep/remove decision |
| 3 — t-Tests | Welch t-test, p-values, Cohen's d, confidence intervals for difference |
| 4 — CIs | Proportion CIs, forest plot, non-overlapping CI interpretation |
| 5 — Correlation | Heatmap, scatter plots, confounders, correlation vs causation |
| 6 — Visualisations | 12 production-quality charts with correct labels and interpretations |
| 7 — NumPy & Pandas | Array conversion, vectorised ops, groupby, apply, pivot tables |
| 8 — Findings | Structured narrative: patterns, outliers, significance, ML implications |
| 9 — Self-Check | Conceptual questions with worked answers |

**Next step:** Week 4 will use this exact cleaned, transformed dataset to train your first classification models.

---
*Week 3 · Statistics for ML & Python Data Stack · Takshashila University*